# Model and trainning

this is where the model is and where to train it. In this notebook, we'll also create modified midi files that went through the model. 

In [ ]:
import torch
import numpy as np
import torch.nn as nn
from torch.nn.functional import softplus

# DATA IMPORTING LIBRARIES
# Add the methods folder to the path
import sys
sys.path.insert(0, '../methods/')

from data.dataloader import MidiDataset
from data.bar_transform import BarTransform
from torch.utils.data import Dataset, DataLoader

from torch.utils.data.sampler import SubsetRandomSampler
from torch.utils.data import DataLoader, random_split

# Visualization and analysis tools (for latent space, if needed later)
import matplotlib.pyplot as plt
from IPython.display import Image, Audio, display, clear_output
import numpy as np
from sklearn.decomposition import PCA

# Matplotlib and seaborn settings
%matplotlib nbagg
%matplotlib inline
import seaborn as sns
sns.set_style("whitegrid")
sns.set_palette(sns.dark_palette("purple"))

from midi_builder import MidiBuilder
builder = MidiBuilder()

# Device configuration
cuda = torch.cuda.is_available()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# MIDI configuration constants
NOTESPERBAR = 16  # Total notes in one bar
totalbars = 16  # Total bars as input
NUM_PITCHES = 60 + 1  # All possible notes to play + 1 for silences

# Derived configuration
TOTAL_NOTES = NOTESPERBAR * totalbars
num_features = NUM_PITCHES  # Size of input feature vector

# Training configuration
batch_size = 64  # Batch size for training
TEACHER_FORCING = True  # Placeholder for teacher forcing (not used yet)


LOAD DATA

In [ ]:
# Configure BarTransform for the desired number of input bars
transform = BarTransform(bars=totalbars, note_count=NUM_PITCHES)

# Import the dataset and apply the transform
midi_dataset = MidiDataset(csv_file='./concat.csv', transform=transform)

# Display memory usage of the dataset
midi_dataset.get_mem_usage()

# Configuration for splitting the dataset
random_seed = 42
test_split = 0.2  # Percentage of dataset reserved for testing
shuffle = True

# Set the random seed for reproducibility
if random_seed is not None:
    np.random.seed(random_seed)

# Determine dataset sizes
dataset_size = len(midi_dataset)           # Total number of MIDI samples in the dataset
test_size = int(test_split * dataset_size) # Number of samples in the test set
train_size = dataset_size - test_size      # Number of samples in the training set

# Split the dataset into training and testing sets
train_dataset, test_dataset = random_split(midi_dataset, [train_size, test_size])

# Create DataLoaders for training and testing
train_loader = DataLoader(train_dataset, shuffle=shuffle, batch_size=batch_size, num_workers=4)
test_loader = DataLoader(test_dataset, shuffle=shuffle, batch_size=batch_size, num_workers=4)

# Display the sizes of the training and testing datasets
print("Train size: {}, Test size: {}".format(train_size, test_size))


MODEL:

In [ ]:
# Define size variables for the network configuration
input_size = NUM_PITCHES

enc_hidden_size = 256  # Hidden size for the encoder
conductor_hidden_size = 256  # Hidden size for the conductor
decoders_hidden_size = 64  # Hidden size for the decoders
decoders_initial_size = 32  # Initial input size for the decoders

n_layers_conductor = 2  # Number of layers for the conductor (not used)
n_layers_decoder = 3  # Number of layers for the decoder (not used)

latent_features = 64  # Dimension of the latent space

sequence_length = 16  # Number of notes per decoder sequence
dropout_rate = 0.2  # Dropout rate for regularization

class VariationalAutoencoder(nn.Module):
    def __init__(self, latent_features, teacher_forcing, eps_i):
        super(VariationalAutoencoder, self).__init__()
        
        self.teacher_forcing = teacher_forcing
        self.eps_i = eps_i
        self.latent_features = latent_features

        # Bi-directional LSTM encoder
        self.encoder = torch.nn.LSTM(
            batch_first=True,
            input_size=input_size,
            hidden_size=enc_hidden_size,
            num_layers=1,
            bidirectional=True
        )
        
        # Linear layer to map encoder output to mean and variance vectors
        self.encoderOut = nn.Linear(
            in_features=enc_hidden_size * 2,  # Bi-directional doubles features
            out_features=latent_features * 2  # Split into mu and sigma
        )
        
        # Linear layer to map latent space to decoder input size
        self.linear_z = nn.Linear(in_features=latent_features, out_features=decoders_initial_size)
        
        self.dropout = nn.Dropout(p=dropout_rate)
        self.worddropout = nn.Dropout2d(p=dropout_rate)
        
        # LSTM conductor and note decoder
        self.conductor = nn.LSTM(decoders_initial_size, decoders_initial_size, num_layers=1, batch_first=True)
        self.decoder = nn.LSTM(NUM_PITCHES + decoders_initial_size, decoders_initial_size, num_layers=1, batch_first=True)
        
        # Linear layer to map decoder output to pitch probabilities
        self.linear = nn.Linear(decoders_initial_size, NUM_PITCHES)

    # Initialize hidden states for the encoder and conductor
    def init_hidden(self, batch_size):
        init = torch.zeros(2, batch_size, enc_hidden_size, device=device)
        c0 = torch.zeros(2, batch_size, enc_hidden_size, device=device)
        init_conductor = torch.zeros(1, batch_size, decoders_initial_size, device=device)
        c_condunctor = torch.zeros(1, batch_size, decoders_initial_size, device=device)
        return init, c0, init_conductor, c_condunctor

    # Scheduled sampling: decide whether to use teacher forcing
    def use_teacher_forcing(self):
        with torch.no_grad():
            tf = np.random.rand(1)[0] <= self.eps_i
        return tf

    # Adjust the probability of teacher forcing
    def set_scheduled_sampling(self, eps_i):
        self.eps_i = eps_i

    # Forward pass
    def forward(self, x):
        batch_size = x.size(0)
        note = torch.zeros(batch_size, 1, NUM_PITCHES, device=device)
        the_input = torch.cat([note, x], dim=1)
        outputs = {}

        # Initialize hidden states
        h0, c0, hconductor, cconductor = self.init_hidden(batch_size)

        # Apply word dropout to input
        x = self.worddropout(x)

        # Encoder: process the input sequence
        x, hidden = self.encoder(x, (h0, c0))

        # Map encoder output to mean and variance vectors
        x = self.encoderOut(x)
        mu, log_var = torch.chunk(x, 2, dim=-1)
        log_var = softplus(log_var)

        # Reparameterization trick
        with torch.no_grad():
            epsilon = torch.randn(batch_size, 1, self.latent_features, device=device)
        sigma = torch.exp(log_var * 2)
        z = mu + epsilon * sigma

        # Map latent space to conductor input
        z = self.linear_z(z)

        conductor_hidden = (hconductor, cconductor)
        counter = 0
        notes = torch.zeros(batch_size, TOTAL_NOTES, NUM_PITCHES, device=device)

        # Decode each latent vector to generate notes
        for i in range(16):
            embedding, conductor_hidden = self.conductor(z[:, i, :].view(batch_size, 1, -1), conductor_hidden)
            if self.use_teacher_forcing():
                decoder_hidden = (torch.randn(1, batch_size, decoders_initial_size, device=device),
                                  torch.randn(1, batch_size, decoders_initial_size, device=device))
                embedding = embedding.expand(batch_size, NOTESPERBAR, embedding.shape[2])
                e = torch.cat([embedding, the_input[:, range(i * 16, i * 16 + 16), :]], dim=-1)
                notes2, decoder_hidden = self.decoder(e, decoder_hidden)
                aux = self.linear(notes2)
                aux = torch.softmax(aux, dim=2)
                notes[:, range(i * 16, i * 16 + 16), :] = aux
            else:
                decoder_hidden = (torch.randn(1, batch_size, decoders_initial_size, device=device),
                                  torch.randn(1, batch_size, decoders_initial_size, device=device))
                for _ in range(sequence_length):
                    e = torch.cat([embedding, note], dim=-1)
                    e = e.view(batch_size, 1, -1)
                    note, decoder_hidden = self.decoder(e, decoder_hidden)
                    aux = self.linear(note)
                    aux = torch.softmax(aux, dim=2)
                    notes[:, counter, :] = aux.squeeze()
                    note = aux
                    counter += 1

        # Store outputs
        outputs["x_hat"] = notes
        outputs["z"] = z
        outputs["mu"] = mu
        outputs["log_var"] = log_var

        return outputs

# Initialize the Variational Autoencoder
net = VariationalAutoencoder(latent_features, TEACHER_FORCING, eps_i=1)

# Transfer model to GPU if available
if cuda:
    net = net.cuda()

print(net)


Optimizer

In [ ]:
# Directly taken from notebook, may require adaptation for specific use case

from torch.nn.functional import binary_cross_entropy
from torch import optim
from torch.distributions.normal import Normal
from torch.distributions.kl import kl_divergence

# Define the Evidence Lower Bound (ELBO) loss function
def ELBO_loss(y, t, mu, log_var, weight):
    """
    Compute the ELBO loss for the VAE.

    Args:
    y (torch.Tensor): Reconstructed output.
    t (torch.Tensor): Target output.
    mu (torch.Tensor): Mean of the latent space distribution.
    log_var (torch.Tensor): Log variance of the latent space distribution.
    weight (float): Weight factor for the KL divergence term (useful for warm-up).

    Returns:
    Tuple containing:
    - ELBO loss
    - Mean KL divergence
    - Weighted KL divergence
    """
    # Reconstruction error: log[p(x|z)]
    # Summed over features
    likelihood = -binary_cross_entropy(y, t, reduction="none")
    likelihood = likelihood.view(likelihood.size(0), -1).sum(1)

    # Regularization error: KL divergence between q(z|x) and prior p(z)
    # Approximate posterior q(z|x) = N(z | mu, sigma^2)
    # Prior p(z) = N(z | 0, I)
    sigma = torch.exp(log_var * 2)
    n_mu = torch.Tensor([0])
    n_sigma = torch.Tensor([1])
    if cuda:
        n_mu = n_mu.cuda()
        n_sigma = n_sigma.cuda()

    # Define normal distributions for the prior (p) and approximate posterior (q)
    p = Normal(n_mu, n_sigma)
    q = Normal(mu, sigma)

    # Compute KL divergence between q and p
    kl_div = kl_divergence(q, p)

    # Alternative analytic KL divergence (currently not used, can be uncommented):
    # kl = -weight * torch.sum(1 + log_var - mu**2 - torch.exp(log_var), dim=(1, 2))

    # Compute ELBO: likelihood + weighted KL divergence
    ELBO = torch.mean(likelihood) - (weight * torch.mean(kl_div))  # Weight applies a warm-up factor

    # Return negative ELBO (to minimize), mean KL divergence, and weighted KL divergence
    return -ELBO, kl_div.mean(), weight * kl_div.mean()

# Define the optimizer for the VAE
# Adam optimizer is well-suited for VAEs
optimizer = optim.Adam(net.parameters(), lr=0.001)

# Assign the ELBO loss function
loss_function = ELBO_loss


Testing if forward pass works

In [ ]:
from torch.autograd import Variable

# Setting dummy data
# Generating dummy data for testing
a = np.random.randint(NUM_PITCHES, size=TOTAL_NOTES)
data = np.zeros((TOTAL_NOTES, NUM_PITCHES))
data[np.arange(TOTAL_NOTES), a] = 1  # One-hot encoding for dummy data

a = np.random.randint(NUM_PITCHES, size=TOTAL_NOTES)
data1 = np.zeros((TOTAL_NOTES, NUM_PITCHES))
data1[np.arange(TOTAL_NOTES), a] = 1  # One-hot encoding for another dummy sample

# Combine dummy data into a batch
d = np.zeros((2, TOTAL_NOTES, NUM_PITCHES))
d[0] = data
d[1] = data1

print(d.shape)  # Should print (2, TOTAL_NOTES, NUM_PITCHES)

# Add an additional dimension to match input format
x = d

# Convert to PyTorch tensor and wrap as a variable
x = Variable(torch.Tensor(x))

# Transfer to GPU if available
if cuda:
    x = x.cuda()

# Running forward pass through the network
outputs = net(x)

# Extract outputs
x_hat = outputs["x_hat"]
mu, log_var = outputs["mu"], outputs["log_var"]
z = outputs["z"]

# Compute loss using ELBO loss function
loss, kl, klw = loss_function(x_hat, x, mu, log_var, 1)

# Print shapes and loss components for debugging
print(f"Input shape: {x.shape}")  # Input shape
print(f"Reconstructed output shape: {x_hat.shape}")  # x_hat shape
print(f"Latent space shape: {z.shape}")  # z shape
print(f"Loss: {loss.item()}")  # Total ELBO loss
print(f"KL Divergence: {kl.item()}")  # Mean KL divergence


## TRAINING

## Show differences in decay strategies

In [ ]:
scheduled_decay_rate = 40

def lin_decay(i, mineps=0):
    return np.max([mineps, 1 - (1/len(train_loader))*i])

def inv_sigmoid_decay(i, rate=40):
    return rate/(rate + np.exp(i/rate))

eps = []
for i in range(len(train_loader)):
    eps_i = inv_sigmoid_decay(i, rate=scheduled_decay_rate)
    eps.append(eps_i)

    
eps2 = []
for i in range(len(train_loader)):
    eps_i = lin_decay(i, 0)
    eps2.append(eps_i)

f, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(eps, color="red")
ax.plot(eps2, color="green")
ax.legend(['Inverse sigmoid decay', 'Linear decay'], prop={'size': 16})

## Define warmup for training

In [ ]:
# Training configuration
# Adjusted epochs for the warmup and main training phase
num_epochs = 100  # Total number of epochs
warmup_epochs = 90  # Number of warmup epochs
pre_warmup_epochs = 10  # Epochs before starting KL warmup

# Linear interpolation for KL warmup scaling
warmup_lerp = 1 / warmup_epochs

# Ensure warmup epochs fit within the total number of epochs
if warmup_epochs > num_epochs - pre_warmup_epochs:
    warmup_epochs = num_epochs - pre_warmup_epochs

# Plot how the KL warmup scaling evolves over epochs
kl_w = 0  # Initial KL weight
kls = []  # List to store KL weights
for e in range(num_epochs):
    if e >= pre_warmup_epochs:
        kl_w = kl_w + warmup_lerp
        if kl_w > 1:
            kl_w = 1.  # Cap the KL weight at 1
    kls.append(kl_w)

# Visualization of the KL warmup
f, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(kls, color="red", label="KL Term Warmup")
ax.legend(prop={'size': 16})
ax.set_title("KL Term Warmup Over Epochs", fontsize=18)
ax.set_xlabel("Epochs", fontsize=14)
ax.set_ylabel("KL Weight", fontsize=14)
plt.show()

est_time = 4 * num_epochs  
print("Estimated time until completion: {:.2f} hours / {:.2f} minutes / {:.2f} seconds".format(
    est_time / 60, est_time, est_time * 60))


## Train

In [ ]:
import time
import os
import math
from torch.autograd import Variable

tmp_img = "tmp_vae_out.png"

warmup_w = 0
eps_i = 1
use_scheduled_sampling = False

train_loss, valid_loss = [], []
train_kl, valid_kl, train_klw = [], [], []

start = time.time()
print("Training epoch {}".format(0))

for epoch in range(num_epochs):
    batch_loss, batch_kl, batch_klw = [], [], []
    net.train()

    for i_batch, sample_batched in enumerate(train_loader):
        x = sample_batched['piano_rolls'].type('torch.FloatTensor').to(device)

        if epoch >= pre_warmup_epochs and use_scheduled_sampling:
            eps_i = inv_sigmoid_decay(i_batch, rate=scheduled_decay_rate)
        net.set_scheduled_sampling(eps_i)

        outputs = net(x)
        x_hat, mu, log_var = outputs['x_hat'], outputs['mu'], outputs['log_var']

        elbo, kl, kl_w = loss_function(x_hat, x, mu, log_var, warmup_w)

        optimizer.zero_grad()
        elbo.backward()
        optimizer.step()

        batch_loss.append(elbo.item())
        batch_kl.append(kl.item())
        batch_klw.append(kl_w.item())

    train_loss.append(np.mean(batch_loss))
    train_kl.append(np.mean(batch_kl))
    train_klw.append(np.mean(batch_klw))

    # Validation
    with torch.no_grad():
        net.eval()
        x = next(iter(test_loader))['piano_rolls'].type('torch.FloatTensor').to(device)

        net.set_scheduled_sampling(1.0)
        outputs = net(x)
        x_hat, mu, log_var, z = outputs['x_hat'], outputs['mu'], outputs['log_var'], outputs["z"]

        elbo, kl, kl_w = loss_function(x_hat, x, mu, log_var, warmup_w)

        x, x_hat, z = x.to("cpu"), x_hat.to("cpu"), z.detach().to("cpu").numpy()
        valid_loss.append(elbo.item())
        valid_kl.append(kl.item())

    if epoch >= pre_warmup_epochs:
        warmup_w = min(warmup_w + warmup_lerp, 1.0)

    if epoch == 0:
        continue

    # Plotting
    f, axarr = plt.subplots(2, 1, figsize=(10, 10))

    ax = axarr[0]
    ax.set_title("ELBO")
    ax.plot(np.arange(epoch + 1), train_loss, color="black")
    ax.plot(np.arange(epoch + 1), valid_loss, color="gray", linestyle="--")
    ax.legend(['Training', 'Validation'])

    ax = axarr[1]
    ax.set_title("Kullback-Leibler Divergence")
    ax.plot(np.arange(epoch + 1), train_kl, color="black")
    ax.plot(np.arange(epoch + 1), valid_kl, color="gray", linestyle="--")
    ax.plot(np.arange(epoch + 1), train_klw, color="blue", linestyle="--")
    ax.legend(['Training', 'Validation', 'Weighted'])

    print(f"Epoch: {epoch}, {time.time() - start:.2f} seconds elapsed")
    plt.savefig(tmp_img)
    plt.close(f)
    display(Image(filename=tmp_img))
    clear_output(wait=True)
    os.remove(tmp_img)

print(f"Finished. Time elapsed: {time.time() - start:.2f} seconds")


# Reprint the graph but skip a few epochs

In [ ]:
skip = 5
save_img = os.path.join("midi","kl_elbo.png")
if skip > num_epochs:
    print("Can't skip more than epochs run.")
    skip = 0

f, axarr = plt.subplots(2, 1, figsize=(10, 10))
    
# Loss
ax = axarr[0]
ax.set_title("ELBO")
ax.set_xlabel('Epoch')
ax.set_ylabel('Error')

ax.plot(np.arange(skip, epoch+1), train_loss[skip:], color="black")
ax.plot(np.arange(skip, epoch+1), valid_loss[skip:], color="gray", linestyle="--")
ax.legend(['Training', 'Validation'])

# KL / reconstruction
ax = axarr[1]

ax.set_title("Kullback-Leibler Divergence")
ax.set_xlabel('Epoch')
ax.set_ylabel('KL divergence')


ax.plot(np.arange(skip, epoch+1), train_kl[skip:], color="black")
ax.plot(np.arange(skip, epoch+1), valid_kl[skip:], color="gray", linestyle="--")
ax.legend(['Training', 'Validation'])

plt.savefig(save_img)
plt.close(f)
display(Image(filename=save_img))

print("Time elapsed: {} seconds".format(end_time))

print("-----")

print("Final KL. Train: {}, Validation: {}".format(train_kl[-1], valid_kl[-1]))
print("Final loss. Train: {}, Validation: {}".format(train_loss[-1], valid_loss[-1]))

# Show reconstructions

In [ ]:
if not os.path.exists('midi'):
    os.makedirs('midi')

x_hat_np = x_hat.detach().numpy()
x_hat_np.shape
for i, seq in enumerate(x_hat_np):
    row_maxes = seq.max(axis=1).reshape(-1, 1)
    midi_out = np.where(seq == row_maxes, 1, 0)
    
    if np.all(midi_out[:,-1]):
        print("Midi: {} is all silent".format(i))
        continue

    np.savetxt("midi/csv_midi_out_{}.csv".format(i), midi_out, delimiter=";")

    midi = builder.midi_from_piano_roll(midi_out[:,:-1]) # Select all notes but the silent one
    plt.figure(figsize=(10, 3))
    plt.title("Midi {}".format(i))
    
    builder.plot_midi(midi)
    plt.savefig("midi/img_midi_{}.png".format(i))

    midi.write('midi/{}.mid'.format(i))

# Compare to originals

In [ ]:
x_np = x.detach().numpy()
x_np.shape
for i, seq in enumerate(x_np):
    midi_out = seq

    if np.all(midi_out[:,-1]):
        print("Midi: {} is all silent".format(i))
        continue
    
    midi = builder.midi_from_piano_roll(midi_out[:,:-1]) # Select all notes but the silent one
    plt.figure(figsize=(10, 3))
    plt.title("Orig Midi {}".format(i))
    
    builder.plot_midi(midi)
    plt.savefig("midi/img_midi_{}_orig.png".format(i))

    midi.write('midi/{}_orig.mid'.format(i))

# Generate from the latent space

In [ ]:
gen_batch = 10
z_gen = torch.randn(gen_batch, totalbars, decoders_initial_size).to(device)  # Sample from latent space

# Initialize hidden states for the conductor and decoder
h_gen, c_gen, hconductor_gen, cconductor_gen = net.init_hidden(gen_batch)
conductor_hidden_gen = (hconductor_gen, cconductor_gen)
decoder_hidden_gen = (torch.randn(1, gen_batch, decoders_initial_size, device=device),
                      torch.randn(1, gen_batch, decoders_initial_size, device=device))

notes_gen = torch.zeros(gen_batch, TOTAL_NOTES, NUM_PITCHES, device=device)  # Generated notes container
note_gen = torch.zeros(gen_batch, 1, NUM_PITCHES, device=device)  # Initial note input

counter = 0

# Generate notes sequentially for each bar
for i in range(totalbars):
    decoder_hidden_gen = (torch.randn(1, gen_batch, decoders_initial_size, device=device),
                          torch.randn(1, gen_batch, decoders_initial_size, device=device))

    # Conductor generates an embedding for the current bar
    embedding_gen, conductor_hidden_gen = net.conductor(z_gen[:, i, :].view(gen_batch, 1, -1), conductor_hidden_gen)

    for _ in range(sequence_length):  # Generate notes for the current bar
        e_gen = torch.cat([embedding_gen, note_gen], dim=-1).view(gen_batch, 1, -1)  # Combine embedding and previous note
        note_gen, decoder_hidden_gen = net.decoder(e_gen, decoder_hidden_gen)  # Decode to generate a new note
        aux_gen = net.linear(note_gen)
        aux_gen = torch.softmax(aux_gen, dim=2)  # Apply softmax to get probabilities

        notes_gen[:, counter, :] = aux_gen.squeeze()  # Store generated note
        note_gen = aux_gen
        counter += 1

notes_gen  # Final generated notes


# Let's visualize

In [ ]:
notes_np = notes_gen.cpu().detach().numpy()
notes_np.shape
for i, seq in enumerate(notes_np):
    row_maxes = seq.max(axis=1).reshape(-1, 1)
    midi_out = np.where(seq == row_maxes, 1, 0)
    if np.all(midi_out[:,-1]):
        print("Midi: {} is all silent".format(i))
        continue

    np.savetxt("midi/gen_csv_midi_out_{}.csv".format(i), midi_out, delimiter=";")

    midi = builder.midi_from_piano_roll(midi_out[:,:-1]) # Select all notes but the silent one
    plt.figure(figsize=(10, 3))
    plt.title("Gen Midi {}".format(i))
    
    builder.plot_midi(midi)
    plt.savefig("midi/gen_img_midi_{}.png".format(i))

    midi.write('midi/gen_{}.mid'.format(i))